# 05 · Phase A: volatility-scaling artifact (Colab T4)

Same 9 equities (SPY + the 8 confirmation stocks/ETFs), same dates; three ways of asking Llama:
**volbins** (ranges scaled by recent volatility — what we used so far), **fixbins** (ranges with one
constant scale per asset), **labels** (up / down / no clear effect). Each with real and shuffled news.

Colab Secrets with *Notebook access* on: `ALPACA_API_KEY`, `ALPACA_SECRET_KEY`, `HF_TOKEN`.
After a disconnect: rerun **Setup**, then the step you were on (everything resumes).

## Setup

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/kronos-agent'
CODE = f'{ROOT}/code'

# Unpack code from Drive. Set UPDATE = True once after uploading a new kronos-agent.zip.
import os, shutil, zipfile, sys
UPDATE = False
ZIP = f'{ROOT}/kronos-agent.zip'
if UPDATE or not os.path.exists(f'{CODE}/kronos_agent/artifact.py'):
    assert os.path.exists(ZIP), f'upload kronos-agent.zip to {ROOT}/ first'
    shutil.rmtree(CODE, ignore_errors=True)
    zipfile.ZipFile(ZIP).extractall(CODE)

%pip install -q ccxt yfinance bitsandbytes accelerate
%pip install -q -e "$CODE" --no-deps
if CODE not in sys.path:
    sys.path.insert(0, CODE)
for k in ('ALPACA_API_KEY', 'ALPACA_SECRET_KEY', 'HF_TOKEN'):
    os.environ[k] = userdata.get(k)

SPY = 'configs/artifact/spy.yaml'
EQ = ' '.join(f'configs/confirm/{t}.yaml' for t in ['qqq', 'iwm', 'aapl', 'msft', 'nvda', 'amzn', 'tsla', 'jpm'])
ALL = SPY + ' ' + EQ
!nvidia-smi -L
print(ROOT, CODE)

## Step 0 · Simulation (CPU, ~2 min) — the mechanism on synthetic data

In [ ]:
!cd $CODE && python scripts/run_artifact.py --simulate --seeds 50

## Step 1 · Show the new prompts for SPY (CPU, no LLM)

In [ ]:
!cd $CODE && python scripts/run_llm.py --origins-only --dry-run --root $ROOT --config $SPY --set llm.mode=fixbins 2>&1 | tail -n 22
!cd $CODE && python scripts/run_llm.py --origins-only --dry-run --root $ROOT --config $SPY --set llm.mode=labels 2>&1 | tail -n 12

## Step 2 · Score (GPU, ~1–1.5 h total; resumable)

In [ ]:
# SPY with the original volatility-scaled ranges (the 8 others already have these from notebook 04)
!cd $CODE && python scripts/run_llm.py --origins-only --variant real shuffled --root $ROOT --config $SPY

In [ ]:
!cd $CODE && python scripts/run_llm.py --origins-only --variant real shuffled --root $ROOT --config $ALL --set llm.mode=fixbins

In [ ]:
!cd $CODE && python scripts/run_llm.py --origins-only --variant real shuffled --root $ROOT --config $ALL --set llm.mode=labels

## Step 3 · Report (CPU, ~1 min)

In [ ]:
!cd $CODE && python scripts/run_artifact.py --root $ROOT --config $ALL